In [31]:
import gensim.downloader
model = gensim.downloader.load("glove-wiki-gigaword-50") # replace with "word2vec-google-news-300" for Google News Word2Vec (Warning: much bigger size!)
print("Vocabulary size:", len(model.key_to_index))

Vocabulary size: 400000


# Embeddings

This notebook explores *word embeddings* — numerical vector representations of words that capture semantic meaning and relationships.

The embeddings used here come from the **"glove-wiki-gigaword-50"** model, a compact and efficient dataset that:
- Uses **50-dimensional** vectors (lightweight and fast to load)
- Captures broad semantic relationships between words
- Is ideal for educational and interactive exploration

### Key Facts
- Each **token** corresponds to one **embedding vector**.
- The notebook includes a **selection of 111 tokens** grouped into **12 semantic categories**.
- The variable `categories` maps category names to the list of tokens belonging to them.

You’ll use these embeddings to:
- Compute **similarities** between tokens
- Explore how **vector arithmetic** reflects linguistic structure

### Categories

In [2]:
categories = {
    "months": [
        "january", "february", "march", "april", "june", "july",
        "august", "september", "october", "november", "december"
    ],
    "seasons": [
        "spring", "summer", "fall", "autumn", "winter"
    ],
    "professions_and_people": [
        "man", "woman", "nurse", "doctor", "uncle", "aunt"
    ],
    "animals": [
        "dog", "cat", "horse", "lion", "cow", "elephant", "tiger", "wolf",
        "squid", "dolphin", "whale", "eagle", "snake", "frog", "monkey",
        "deer", "rabbit"
    ],
    "vehicles": [
        "car", "bicycle", "motorcycle", "bus", "truck", "van", "train",
        "tram", "subway", "airplane", "helicopter", "boat", "ship",
        "submarine", "rocket", "scooter", "tractor", "tank", "skateboard"
    ],
    "vegetables": [
        "carrot", "broccoli", "spinach", "potato", "tomato",
        "corn", "cucumber", "banana", "pumpkin"
    ],
    "colors": [
        "red", "green", "blue", "yellow", "white", "black", "orange", "purple", "brown"
    ],
    "nature": [
        "sky", "grass", "plant", "tree", "sun", "ocean", "leaf", "fire", "blood"
    ],
    "food": [
        "apple", "banana", "orange", "strawberry", "grape", "fruit",
        "pastry", "bread", "cake", "pie", "cookie", "rose"
    ],
    "education": [
        "school", "book", "pencil", "desk", "teacher", "backpack", "science"
    ],
    "sports": [
        "sport", "ball", "team", "goal", "run", "coach"
    ],
    "art": [
        "art", "paint", "brush", "color", "canvas", "draw"
    ]
}

from IPython.display import display, HTML

# Generate formatted HTML
html_parts = []
for cat, toks in categories.items():
    title = cat.replace("_", " ").title()
    token_list = ", ".join(toks)
    html_parts.append(f"<b>{title}</b>: <span>{token_list}</span><br>")

# Combine and display
display(HTML("\n".join(html_parts)))

### Cosine Similarity

Embeddings are vectors in a high-dimensional space.  
To measure how *similar* two embeddings are, we use **cosine similarity**, which computes the cosine of the angle between two vectors:

$$
\text{similarity}(A, B) = \frac{A \cdot B}{||A|| \, ||B||}
$$

- A value of **1.0** means the vectors point in the *same direction* (high similarity).  
- A value near **0** means they’re *unrelated* (orthogonal).  
- A negative value means they’re *opposite in meaning* (rare in GloVe).  

While cosine similarity captures meaningful relations between words, remember:
> These static embeddings are just the *starting point* for large language models (LLMs).  
> Modern LLMs add multiple **layers of attention and reasoning**, allowing them to infer nuanced context, relationships, and syntax beyond what embeddings alone can represent.
---

### 🧩 Exercise 1 — Exploring Similarities

Below, you’ll find an interactive tool that:
- Lets you enter a token  
- Lists the **five most similar tokens** by cosine similarity  
- Allows filtering by **categories** - deselect all categories except the ones you want to explore

Try the following:
1. Explore how similarity changes across categories:  
   - `milk` → in category **animals**  
   - `fast` → in category **vehicles**  
   - `green` → in category **vegetables**

In [3]:
"""
Interactive Similarity Explorer (Gensim-based)
----------------------------------------------
Uses a preloaded Gensim model, e.g.:
    model = gensim.downloader.load("glove-wiki-gigaword-50")

Lets you:
- Search for a token and find its most similar words.
- Restrict the search to selected semantic categories.
"""

import ipywidgets as widgets
from IPython.display import display, clear_output
from difflib import SequenceMatcher

# ---------- HELPERS ----------
def levenshtein_like(a, b):
    """Approximate text similarity ratio (for spelling suggestions)."""
    return SequenceMatcher(None, a, b).ratio()

# Build category lookup (optional, for filtering)
token_to_cat = {t: cat for cat, toks in categories.items() for t in toks if t in model.key_to_index}

# ---------- INTERACTIVE EXPLORER ----------
def create_interactive_explorer():
    search_box = widgets.Text(
        placeholder="Type a token...",
        description="Token:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="350px")
    )

    category_checkboxes = {
        cat: widgets.Checkbox(
            value=True,
            description=cat.replace("_", " ").title(),
            indent=False,
            layout=widgets.Layout(width="200px")
        )
        for cat in categories
    }

    checkbox_grid = widgets.GridBox(
        list(category_checkboxes.values()),
        layout=widgets.Layout(grid_template_columns="repeat(3, 200px)")
    )

    output = widgets.Output()

    def on_submit(change):
        token = change["new"].strip()
    
        with output:
            clear_output()
    
            # Validate token
            if token not in model.key_to_index:
                print(f"⚠️ Token '{token}' not found in model vocabulary.")
                return
    
            # Collect allowed tokens from selected categories
            active_cats = [c for c, cb in category_checkboxes.items() if cb.value]
            allowed_tokens = {t for c in active_cats for t in categories[c] if t in model.key_to_index and t != token}
    
            if not allowed_tokens:
                print("⚠️ No valid tokens found in selected categories.")
                return
    
            # Compute cosine similarity for all allowed tokens
            results = []
            for t in allowed_tokens:
                try:
                    sim = model.similarity(token, t)
                    results.append((t, sim))
                except KeyError:
                    continue
    
            # Sort by descending similarity
            results.sort(key=lambda x: x[1], reverse=True)
            results = results[:5]  # top 5
    
            if not results:
                print("⚠️ No results in selected categories.")
                return
    
            # Display
            print(f"Nearest neighbors for '{token}' (filtered by categories):\n")
            for t, sim in results:
                cat = token_to_cat.get(t, "unknown").replace("_", " ").title()
                print(f"  {t:15s}  similarity = {sim:.3f}   [{cat}]")

    # Observe user input
    search_box.observe(on_submit, names="value")

    # Update when checkboxes change
    def update_output(*_):
        if search_box.value.strip():
            on_submit({"new": search_box.value})
    for cb in category_checkboxes.values():
        cb.observe(update_output, names="value")

    display(widgets.VBox([
        search_box,
        widgets.Label("Select categories to include in the search:"),
        checkbox_grid,
        output
    ]))

# --- Display the interactive explorer ---
create_interactive_explorer()

## Relationships Between Embeddings

Word embeddings are **compositional** — meaning relationships between words are often reflected as *directions* or *offsets* in the vector space.

This allows for **vector arithmetic** such as:
$$
\text{king} - \text{man} + \text{woman} \approx \text{queen}
$$

You can try similar combinations below to explore semantic relationships.

---

### 🧩 Exercise 2 — Vector Arithmetic

Use the cell below to add or subtract embeddings and see which tokens are closest to the resulting vector. Try examples like:

- `father - man + woman` → ?  
- `man + medicine` → ?  
- `ios + google` → ?  
- `walking + car` → ?
- `sushi - japan + germany` → ?

In [4]:
vec = model["father"] - model["man"] + model["woman"]

# Find most similar tokens to this vector
results = model.most_similar(positive=[vec], topn=10)
print("Top 10 most similar tokens:\n")
for word, score in results:
    print(f"  {word:15s}  similarity = {score:.3f}")

Top 10 most similar tokens:

  cats             similarity = 0.924
  dogs             similarity = 0.915
  dog              similarity = 0.824
  animals          similarity = 0.789
  rabbits          similarity = 0.761
  horses           similarity = 0.740
  goats            similarity = 0.735
  cows             similarity = 0.733
  pigs             similarity = 0.733
  herd             similarity = 0.732


---

### 🧩 Exercise 3 — Context and Composition

Consider the sentence:  
> “The fox was quick, it jumped across the fence.”

1. Check the most similar tokens to **"it"**.  
2. Now imagine the model had access to context — it might realize “it” refers to “fox” and add some of the embeddings of “fox” to “it” as a result.  
   Add the embeddings of `"it"` and `"fox"` together, and search for similar tokens again.  
3. Observe how the resulting vector moves closer to tokens related to “fox”.

> This illustrates how contextualized embeddings (used in modern LLMs) dynamically adjust meaning depending on context — something static embeddings can only approximate.
> 
> In the next section, we’ll explore **Attention**, the mechanism that enables this context sensitivity. Unlike our simple vector addition, attention doesn’t combine meanings linearly — it learns to weight and integrate information dynamically across all tokens in a sequence.

In [34]:
vec = model["it"]

print("Top 10 most similar tokens:\n")
for word, score in model.most_similar(positive=[vec], topn=10):
    print(f"  {word:15s}  similarity = {score:.3f}")

Top 10 most similar tokens:

  it               similarity = 1.000
  this             similarity = 0.944
  but              similarity = 0.929
  .                similarity = 0.925
  same             similarity = 0.925
  so               similarity = 0.919
  now              similarity = 0.915
  that             similarity = 0.913
  though           similarity = 0.913
  because          similarity = 0.911
